# Status report

By Ben Welsh

Generates basic statistics from [News Homepages database extracts](https://palewi.re/docs/news-homepages/extracts.html).

In [ ]:
import json
import pandas as pd
import altair as alt
from pathlib import Path
from datetime import datetime, timedelta

In [ ]:
alt.renderers.enable('altair_saver', fmts=['vega-lite', 'png'])

In [ ]:
this_dir = Path("__file__").parent.absolute()

In [ ]:
sources_dir = this_dir.parent / "newshomepages" / "sources"

In [ ]:
extracts_dir = this_dir.parent / "extracts" / "csv"

In [ ]:
df = pd.read_csv(
    extracts_dir / "screenshot-files.csv",
    parse_dates=["mtime"],
    usecols=["identifier", "handle", "file_name", "mtime"]
)

In [ ]:
df['date'] = df.mtime.dt.date

In [ ]:
df["date"] = pd.to_datetime(df["date"])

How many total sites?

In [ ]:
total_sites = len(df.handle.unique())

In [ ]:
total_sites

How many total screenshots?

In [ ]:
total_screenshots = len(df)

In [ ]:
total_screenshots

When did we start?

In [ ]:
start_date = min(df.date)

In [ ]:
start_date

How many screenshots in the last week?

In [ ]:
today = datetime.now().date()

In [ ]:
today

In [ ]:
one_week_ago = today - timedelta(days=7)

In [ ]:
one_week_ago

In [ ]:
df_this_week = df[df.date > pd.to_datetime(one_week_ago)]

In [ ]:
screenshots_this_week = len(df_this_week)

In [ ]:
screenshots_this_week

Write out data points

In [ ]:
output = dict(
    total_sites=total_sites,
    total_screenshots=total_screenshots,
    screenshots_this_week=screenshots_this_week,
)

In [ ]:
json.dump(output, open(this_dir / 'status-report.json', 'w'), indent=2)

Chart the number of sites by date

In [ ]:
sites_by_date = df[['date', 'handle']].drop_duplicates().groupby("date").size().rename("sites").reset_index().sort_values("date")

In [ ]:
sites_by_date['rolling_mean'] = sites_by_date.sites.rolling(7).mean()

In [ ]:
chart = alt.Chart(
    sites_by_date,
    title="Sites archived by @newshomepages",
    width=500
)

bars = chart.mark_bar(
    fill="#cecece"
).encode(
    x=alt.X("date:T", title="Date", timeUnit="yearmonthdate", axis=alt.Axis(format="%B %d", grid=False)),
    y=alt.Y("sites:Q", title="Sites"),
)

line = chart.mark_line(color='#727272', strokeWidth=3).encode(
    x='date:T',
    y='rolling_mean:Q'
)

label = chart.encode(
    x=alt.X('max(date):T'),
    y=alt.Y('rolling_mean:Q', aggregate=alt.ArgmaxDef(argmax='date')),
    text='rolling_mean'
)

# Create a text label
text = label.mark_text(align='left', dx=4)

# Create a circle annotation
circle = label.mark_circle(size=75, color="#727272")

(bars + line + circle).save(this_dir / 'sites-by-date.svg')

Chart the number of screenshots by date

In [ ]:
screenshots_by_date = df.groupby("date").size().rename("screenshots").reset_index().sort_values("date")

In [ ]:
screenshots_by_date['rolling_mean'] = screenshots_by_date.screenshots.rolling(7).mean()

In [ ]:
chart = alt.Chart(
    screenshots_by_date,
    title="Screenshots saved by @newshomepages",
    width=500
)

bars = chart.mark_bar(
    fill="#cecece"
).encode(
    x=alt.X("date:T", title="Date", timeUnit="yearmonthdate", axis=alt.Axis(format="%B %d", grid=False)),
    y=alt.Y("screenshots:Q", title="Screenshots"),
)

line = chart.mark_line(color='#727272', strokeWidth=3).encode(
    x='date:T',
    y='rolling_mean:Q'
)

label = chart.encode(
    x=alt.X('max(date):T'),
    y=alt.Y('rolling_mean:Q', aggregate=alt.ArgmaxDef(argmax='date')),
    text='rolling_mean'
)

# Create a text label
text = label.mark_text(align='left', dx=4)

# Create a circle annotation
circle = label.mark_circle(size=75, color="#727272")

(bars + line + circle).save(this_dir / 'screenshots-by-date.png')